# Fermionic operators

This notebook introduces fermionic creation and annihilation operators with OpenFermion. We use the interleaved spin-orbital ordering $(0_\alpha,0_\beta,1_\alpha,1_\beta,\ldots)$. The canonical anticommutation relations are

$$\{a_p,a_q^\dagger\}=\delta_{pq},\qquad \{a_p,a_q\}=\{a_p^\dagger,a_q^\dagger\}=0.$$

In [1]:
from openfermion import FermionOperator
from openfermion import hermitian_conjugated, normal_ordered

## Slater determinants

A product of creation operators acting on the vacuum defines an occupation-number state, or Slater determinant,

$$|s_0\rangle=a_0^\dagger a_1^\dagger a_2^\dagger a_3^\dagger|\mathrm{vac}\rangle. $$
$$|s_1\rangle=a_0^\dagger a_1^\dagger a_2^\dagger a_4^\dagger|\mathrm{vac}\rangle. $$

The order of fermionic operators matters because exchanging two creation operators changes the sign. The two states in the code differ by their occupied spin orbitals.

In [2]:
# a b a b a b
s0 = FermionOperator('0^ 1^ 2^ 3^')
s1 = FermionOperator('0^ 1^ 2^ 4^')

## Inner products by normal ordering

To evaluate $\langle s_i|s_j\rangle$, the code takes the Hermitian conjugate of the first operator string, multiplies the strings, and normal-orders the result. The scalar coefficient of the identity term is the vacuum expectation value. Orthogonal occupation patterns give zero, while a normalized determinant gives one.

In [3]:
term = normal_ordered(hermitian_conjugated(s0) * s0)
print('<s0|s0> = ', term.terms.get((), 0.0))

term = normal_ordered(hermitian_conjugated(s1) * s0)
print('<s1|s0> = ', term.terms.get((), 0.0))

<s0|s0> =  1.0
<s1|s0> =  0.0


## Particle number and spin projection

The number operator and z-component of spin are

$$\hat N=\sum_p a_p^\dagger a_p,\qquad \hat S_z=\frac{1}{2}\sum_p \sigma_p a_p^\dagger a_p, $$

where $\sigma_p=+1$ for an alpha spin orbital and $-1$ for a beta spin orbital. Their expectation values identify the electron number and spin projection of each determinant.

In [9]:
# number operator
n = (FermionOperator('0^ 0') +
     FermionOperator('1^ 1') +
     FermionOperator('2^ 2') +
     FermionOperator('3^ 3') +
     FermionOperator('4^ 4'))

term = normal_ordered(hermitian_conjugated(s0) * n * s0)
print('<s0|n|s0>  = ', term.terms.get((), 0.0))
term = normal_ordered(hermitian_conjugated(s1) * n * s1)
print('<s1|n|s1>  = ', term.terms.get((), 0.0))

<s0|n|s0>  =  4.0
<s1|n|s1>  =  4.0


In [10]:
# Sz operator
Sz = 0.5 * (FermionOperator('0^ 0') # alpha
            - FermionOperator('1^ 1') # beta
            + FermionOperator('2^ 2') # alpha
            - FermionOperator('3^ 3') # beta
            + FermionOperator('4^ 4') # alpha
)

term = normal_ordered(hermitian_conjugated(s0) * Sz * s0)
print('<s0|Sz|s0> = ', term.terms.get((), 0.0))

term = normal_ordered(hermitian_conjugated(s1) * Sz * s1)
print('<s1|Sz|s1> = ', term.terms.get((), 0.0))

<s0|Sz|s0> =  0.0
<s1|Sz|s1> =  1.0


## Total spin operator

The total-spin operator is assembled from the spin projection and spin raising/lowering operators,

$$\hat S^2=\hat S_z^2+\frac{1}{2}(\hat S_+\hat S_-+\hat S_-\hat S_+). $$

For a spin eigenstate, $S^2$ has eigenvalue $S(S+1)$. The expectation values below therefore identify the first determinant as singlet-like ($S=0$) and the second as triplet-like ($S=1$).

In [7]:
S_plus = (FermionOperator('1^ 0') + # alpha -> beta
          FermionOperator('3^ 2') + # alpha -> beta
          FermionOperator('5^ 4')   # alpha -> beta
)

S_minus = (FermionOperator('0^ 1') + # beta -> alpha
           FermionOperator('2^ 3') + # beta -> alpha
           FermionOperator('4^ 5')   # beta -> alpha
)

# total spin operator
S2 = Sz * Sz + 0.5 * (S_plus * S_minus + S_minus * S_plus)

In [8]:
term = normal_ordered(hermitian_conjugated(s0) * S2 * s0)
print('<s0|S2|s0> = ', term.terms.get((), 0.0))

term = normal_ordered(hermitian_conjugated(s1) * S2 * s1)
print('<s1|S2|s1> = ', term.terms.get((), 0.0))

<s0|S2|s0> =  0.0
<s1|S2|s1> =  2.0


# Pauli Operators (qubit operators)

$$
I = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}
\qquad
X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}
\qquad
Y = \begin{pmatrix} 0 & -i \\ i & 0 \end{pmatrix}
\qquad
Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}
$$

In [16]:
from openfermion import QubitOperator, get_sparse_operator

op = QubitOperator('X0')

print(op)



1.0 [X0]


## Matrix representation


In [27]:
from openfermion import get_sparse_operator

X = get_sparse_operator(op).toarray()

print(X)


[[0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j]]


## Pauli strings

Tensor product of pauli operators acting on a different site form a Pauli string

$P=X^{(0)} \otimes Y^{(1)} \otimes I^{(2)} \otimes Z^{(3)}$

In [40]:
from openfermion import QubitOperator

P = QubitOperator('X0 Y1 Z3')

print(P)

1.0 [X0 Y1 Z3]


### Matrix representation of pauli strings

$$
P = X \otimes Z =
\begin{pmatrix}
0 & 1 \\
1 & 0
\end{pmatrix}
\otimes
\begin{pmatrix}
1 & 0 \\
0 & -1
\end{pmatrix}
=
\begin{pmatrix}
0 & 0 & 1 & 0 \\
0 & 0 & 0 & -1 \\
1 & 0 & 0 & 0 \\
0 & -1 & 0 & 0
\end{pmatrix}
$$


In [49]:
from openfermion import get_sparse_operator

P = QubitOperator('X0 Z1')
M = get_sparse_operator(P).toarray()

print(M.real)

[[ 0.  0.  1.  0.]
 [ 0.  0.  0. -1.]
 [ 1.  0.  0.  0.]
 [ 0. -1.  0.  0.]]


## Operating with qubit operators

$$
A = \sum_i c_i P_i,
$$

where $P_i$ is a pauli operator.

In [51]:
A = 0.5 * QubitOperator('X0') + 1.2 * QubitOperator('Z1') - 0.7 * QubitOperator('X0 Z1')
print(A)

0.5 [X0] +
-0.7 [X0 Z1] +
1.2 [Z1]


- Check the matrix representation of other operators Y, Z
- Understand the matrix representation of a pauli string (e.g. $X \otimes Y$)